Figure 3 h

This code was written using this library versions 
NumPy: 1.26.4
Pandas: 3.0.3
SciPy: 1.13.1
Matplotlib: 3.10.8
Seaborn: 0.13.2

In [ ]:
#Libraries
import os
import glob
import numpy as np
import pandas as pd
import scipy as scipy
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ===================================================
# Data Loading 
# ===================================================

# Directory parameters
BASE_DIR = r"C:\Users\pdelf\OneDrive\Documents\Proyecto Tejido Cardiaco\Grupos"
TARGET_GROUPS = ["Veh", "Ccpe"]

raw_records = []
sample_global_id = 1  # Continuous numbering across treatments

#Open the csvs and create the database
for group_name in sorted(os.listdir(BASE_DIR)): #search for target folders
    if group_name not in TARGET_GROUPS:
        continue

    group_path = os.path.join(BASE_DIR, group_name)
    csv_folder = os.path.join(group_path, "cvs") #Csvs in this folder

    if os.path.exists(csv_folder):
        csv_files = sorted(glob.glob(os.path.join(csv_folder, "*.csv"))) #one csv per sample
        for file in csv_files:
            original_sample_name = os.path.basename(file).replace(".csv", "")

            try:
                df_temp = pd.read_csv(file)
                tension_col = [
                    col for col in df_temp.columns
                    if col.lower() in ["tension"] #Search for tension column
                ]

                if tension_col:
                    vals = df_temp[tension_col[0]].dropna().values #shouldnt find Nan 
                    if len(vals) > 0:
                        for v in vals:
                            raw_records.append({
                                "Group": group_name,
                                "Sample_ID": sample_global_id,
                                "Sample_Name": original_sample_name,
                                "Tension": v
                            })
                        sample_global_id += 1
            except Exception as e:
                print(f"Error reading {file}: {e}")

# Consolidated raw DataFrame
df_raw = pd.DataFrame(raw_records)

# ===================================================
# Statistical Analysis
# ===================================================

#Objective 1: Characterize global tissue mechanics rather than per-organism averages.

#An edge on each tissue is considered as the unit of analysis, treated as an individual
#observation to preserve intra-sample variability, extreme values and distribution shape.
#Normality and Kolmogorov Smirnov (KS) tests were run on the complete pooled raw dataset 
#to evaluate the global distribution profile and assess overall heterogeneity across conditions.

#Objetive 2: Between-Group Hypothesis Testing

#To compare treatmentes while preserving biological independence (avoiding pseudoreplication), 
# we used the median tension value of each sample (csv) as the representative biological
# unit for non-parametric hypothesis testing (Mann-Whitney U).

print("\nStatistical Analysis")

# ---------------------------------------------------

# Shapiro-Wilk Normality Test
print("\nShapiro-Wilk Normality Test")

shapiro_results = []

for grp in TARGET_GROUPS:
  
  # Extract and pool ALL raw tension data points for the group into one 1D array
  raw_points = df_raw[df_raw["Group"] == grp]["Tension"].dropna().values
  n_points = len(raw_points) 

  if n_points >= 3:
    stat_raw, p_raw = stats.shapiro(raw_points)
    subsample_note = ""

    shapiro_results.append({
        "Group": grp,
        "Total_Raw_Points": n_points,
        "Shapiro_Stat": stat_raw,
        "Shapiro_p_value": p_raw,
        "Normal_p>0.05": p_raw > 0.05,
        "Note": subsample_note.strip(),
    })
  else:
    shapiro_results.append({
        "Group": grp,
        "Total_Raw_Points": n_points,
        "Shapiro_Stat": np.nan,
        "Shapiro_p_value": np.nan,
        "Normal_p>0.05": False,
        "Note": "Insufficient points (<3)",
    })

df_shapiro_raw = pd.DataFrame(shapiro_results)
print(df_shapiro_raw.to_string(index=False))

# --------------------------------------------------------

# Two-Sample Kolmogorov-Smirnov Test
print("\nTwo-Sample Kolmogorov-Smirnov Test")

# Extract raw point arrays by group
veh_raw = df_raw[df_raw["Group"] == "Veh"]["Tension"].dropna().values
ccpe_raw = df_raw[df_raw["Group"] == "Ccpe"]["Tension"].dropna().values

ks_stat, p_ks = stats.ks_2samp(ccpe_raw, veh_raw, alternative="two-sided")


print(f" • Group A: Ccpe (n={len(ccpe_raw)}), Group B: Veh (n={len(veh_raw)})")
print(f" • KS Statistic (D)               : {ks_stat:.4f}")
print(f" • p-value                        : {p_ks:.4e}")
print(f" • Significant (p<0.05)           : {p_ks < 0.05}")
print(f" --------------------------------------------------")

# ---------------------------------------------------

# Aggregate tension per individual sample (biological replicates)
sample_metrics = []

for (group, sample_id), sub_df in df_raw.groupby(["Group", "Sample_ID"]):
    vals = sub_df["Tension"].values
    sample_metrics.append({
        "Group": group,
        "Sample_ID": sample_id,
        "Sample_Name": sub_df["Sample_Name"].iloc[0],
        "Points": len(vals),
        "Mean_Tension": np.mean(vals),
        "Median_Tension": np.median(vals),
        "Std_Dev": np.std(vals, ddof=1)
    })

df_samples = pd.DataFrame(sample_metrics)

# ---------------------------------------------------

# Mann-Whitney U Test
print("\n Mann-Whitney U Test")

#Obtain median values per group, per sample
veh_medians = df_samples[df_samples["Group"] == "Veh"]["Median_Tension"]
ccpe_medians = df_samples[df_samples["Group"] == "Ccpe"]["Median_Tension"]

# Test: Mann-Whitney U Test (Non-parametric) - Both directions
n_ccpe = len(ccpe_medians)
n_veh = len(veh_medians)
#n_total_comb = n_ccpe * n_veh

# Direction: Ccpe vs Veh
stat_mw_ccpe, p_mw_ccpe = stats.mannwhitneyu(ccpe_medians, veh_medians, alternative="two-sided")

print(f" • Group A: Ccpe (n={n_ccpe}), Group B: Veh (n={n_veh})")
print(f" • MW Statistic (U), Ccpe vs Veh  : {stat_mw_ccpe:.4f}")
print(f" • p-value                        : {p_mw_ccpe:.4e}")
print(f" • Significant (p<0.05)           : {p_mw_ccpe < 0.05}")

In [ ]:
# =============================================================================
# Figure 3h
# =============================================================================

# Filter raw dataset for target groups
df_plot = df_raw[df_raw['Group'].isin(['Veh', 'Ccpe'])].copy()

# Custom palette mapping
custom_palette = {
    "Veh": (0.2, 0.45, 0.8),
    "Ccpe": (0.95, 0.75, 0.1)
}

# Visual style configuration
sns.set_theme(style="white")

# =============================================================================
# Plot of the gaussian distributions fitted to the histograms of relative mechanical tension 
# for the vehicle and cCpE conditions. 
# =============================================================================
plt.figure(figsize=(10, 6))

sns.kdeplot(
    data=df_plot, 
    x="Tension", 
    hue="Group", 
    hue_order=["Veh", "Ccpe"],
    fill=True, 
    common_norm=False, 
    palette=custom_palette, 
    alpha=0.2, 
    linewidth=2.5,
    cut=3
)

plt.xlabel("Tension", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.xlim(left=0)  # Constrain x-axis at zero
plt.tight_layout()
plt.savefig("option2_overlapping_kde.png", dpi=300)
plt.show()